# Module 1: Databricks Fundamentals

## Learning Objectives
By the end of this module, you will understand:
- What is Databricks and how to use it
- Databricks File System (DBFS) and how to work with files
- The anatomy of PySpark code
- How to create your first DataFrames
- Understanding SparkSession and Spark Context

---

## 1. What is Databricks?

**Databricks** is a unified analytics platform built on Apache Spark. It provides:
- Easy-to-use notebooks (like Jupyter but more powerful)
- Collaborative workspace for teams
- Managed Apache Spark clusters
- Built-in ML libraries and tools
- Delta Lake (a storage layer that brings ACID transactions to data lakes)

### Why use Databricks?
1. **No setup required** - Clusters are managed for you
2. **Scalability** - Handle big data seamlessly
3. **Collaboration** - Share notebooks and results with teams
4. **Multiple languages** - Python, SQL, Scala, R all supported

### Databricks Community Edition
- **Free version** available at databricks.com
- Perfect for learning and small projects
- Includes a pre-configured cluster
- Limited to individual use

---

## 2. Databricks File System (DBFS)

DBFS is a virtual file system that allows you to interact with:
- Cloud storage (S3, Azure Blob Storage, GCS)
- Local files on the cluster
- Network locations

### DBFS Paths
DBFS paths start with `/dbfs/`:
- `/dbfs/` - Root of the DBFS
- `/dbfs/user/hive/warehouse/` - Default data warehouse location
- `/dbfs/FileStore/` - Location to upload and download files
- `/dbfs/mnt/` - Mounted external storage

### Common DBFS Operations

In [ ]:
# List files in DBFS
dbutils.fs.ls("/dbfs/")

**Note:** `dbutils` is a Databricks utility object automatically available in Databricks notebooks.

### Common dbutils.fs commands:
- `dbutils.fs.ls(path)` - List files in a directory
- `dbutils.fs.head(file_path)` - Show first few lines of a file
- `dbutils.fs.put(file_path, content)` - Write content to a file
- `dbutils.fs.rm(file_path)` - Delete a file or directory
- `dbutils.fs.mv(source, dest)` - Move/rename a file
- `dbutils.fs.cp(source, dest)` - Copy a file

---

## 3. Anatomy of PySpark Code

Every PySpark application follows a standard structure. Let's break it down:

### Step 1: Create a SparkSession
A **SparkSession** is your entry point to Spark functionality. It replaces the older SparkContext.

```python
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MyApp") \
    .getOrCreate()
```

- `SparkSession.builder` - Starts the builder pattern
- `.appName("MyApp")` - Sets the application name (appears in Spark UI)
- `.getOrCreate()` - Gets existing session or creates a new one

### Step 2: Load Data
```python
df = spark.read.csv("path/to/file.csv", header=True)
```

### Step 3: Process Data
```python
df_filtered = df.filter(df["age"] > 18)
```

### Step 4: Show Results
```python
df_filtered.show()
```

### Step 5: Save Results (Optional)
```python
df_filtered.write.csv("path/to/output")
```

---

## 4. Creating Your First DataFrames

A **DataFrame** is PySpark's fundamental data structure. It's similar to:
- Pandas DataFrame
- SQL Table
- Excel spreadsheet

### Why DataFrames?
- ✅ Distributed across cluster nodes for parallel processing
- ✅ Optimized for big data operations
- ✅ Can run SQL queries on them
- ✅ Lazy evaluation (operations not executed until needed)

### Method 1: Create from Python data structures

In [ ]:
# In Databricks, spark is already available
# In local environments, you would need to create it first:
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("MyApp").getOrCreate()

# Method 1a: Create from list of dictionaries
data = [
    {"name": "Alice", "age": 28, "salary": 70000},
    {"name": "Bob", "age": 32, "salary": 85000},
    {"name": "Charlie", "age": 25, "salary": 60000},
    {"name": "Diana", "age": 35, "salary": 95000}
]

df_from_dict = spark.createDataFrame(data)
df_from_dict.show()

**Output:**
```
+-------+---+------+
|   name|age|salary|
+-------+---+------+
|  Alice| 28| 70000|
|    Bob| 32| 85000|
|Charlie| 25| 60000|
|  Diana| 35| 95000|
+-------+---+------+
```

In [ ]:
# Method 1b: Create from list of tuples with explicit schema
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Define the schema (structure) of the data
schema = StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("salary", IntegerType(), True)
])

# Create data as tuples
data_tuples = [
    ("Alice", 28, 70000),
    ("Bob", 32, 85000),
    ("Charlie", 25, 60000),
    ("Diana", 35, 95000)
]

df_from_tuples = spark.createDataFrame(data_tuples, schema=schema)
df_from_tuples.show()

**Output:** Same as above - now with explicitly defined data types

**What's a Schema?**
A schema defines the structure of your DataFrame:
- Column names
- Data types (StringType, IntegerType, DoubleType, etc.)
- Nullable (can column have null values?)

---

In [ ]:
# Method 2: Create from Pandas DataFrame
import pandas as pd

# Create a Pandas DataFrame
pandas_df = pd.DataFrame({
    "name": ["Emma", "Frank", "Grace"],
    "age": [29, 31, 26],
    "department": ["Engineering", "Sales", "Marketing"]
})

# Convert to Spark DataFrame
spark_df = spark.createDataFrame(pandas_df)
spark_df.show()

**Output:**
```
+-----+---+----------+
| name|age|department|
+-----+---+----------+
| Emma| 29|Engineering|
|Frank| 31|     Sales|
|Grace| 26| Marketing|
+-----+---+----------+
```

---

## 5. Understanding SparkContext vs SparkSession

### SparkContext (Older)
- Low-level API for Spark
- Works with RDDs (Resilient Distributed Datasets)
- Less user-friendly

### SparkSession (Recommended)
- High-level API
- Works with DataFrames
- Better performance optimizations
- SQL support built-in

### Accessing SparkContext from SparkSession

In [ ]:
# Get SparkContext from SparkSession
sc = spark.sparkContext

# Get SparkConf
conf = sc.getConf()

# Print some info
print(f"App Name: {conf.get('spark.app.name')}")
print(f"Master: {conf.get('spark.master')}")
print(f"Spark Version: {spark.version}")

**Output (in Databricks):**
```
App Name: Databricks Shell
Master: local[*]
Spark Version: 11.3.x
```

---

## 6. DataFrame Properties and Metadata

Once you have a DataFrame, you can inspect its properties:

In [ ]:
# Create a sample DataFrame
data = [
    ("Alice", 28, 70000.50),
    ("Bob", 32, 85000.75),
    ("Charlie", 25, 60000.00)
]
columns = ["name", "age", "salary"]
df = spark.createDataFrame(data, schema=columns)

# Get the schema
print("Schema:")
df.printSchema()

print("\nColumn Names:")
print(df.columns)

print("\nData Types:")
print(df.dtypes)

print("\nNumber of Rows:")
print(df.count())

print("\nNumber of Columns:")
print(len(df.columns))

**Output:**
```
Schema:
root
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)
 |-- salary: double (nullable = true)

Column Names:
['name', 'age', 'salary']

Data Types:
[('name', 'string'), ('age', 'long'), ('salary', 'double')]

Number of Rows:
3

Number of Columns:
3
```

---

## 7. Lazy Evaluation - Key Concept!

**This is crucial to understand:** Spark uses **lazy evaluation**

### What does lazy evaluation mean?
- When you write a transformation, it's not executed immediately
- Spark builds a logical plan of operations
- Only when you call an **action**, the actual computation happens

### Transformations vs Actions

**Transformations** (lazy - not executed):
- `filter()` - Filter rows
- `select()` - Select columns
- `map()` - Apply function to each element
- `join()` - Combine DataFrames
- `groupBy()` - Group data

**Actions** (eager - executed immediately):
- `show()` - Display DataFrame
- `collect()` - Return all data to driver
- `count()` - Count rows
- `first()` - Get first row
- `write.csv()` - Save to file

### Example

In [ ]:
data = [
    ("Alice", 28),
    ("Bob", 32),
    ("Charlie", 25),
    ("Diana", 35)
]
df = spark.createDataFrame(data, ["name", "age"])

# TRANSFORMATION (not executed yet)
# Spark doesn't filter anything here - just plans it
filtered_df = df.filter(df["age"] > 28)
print("Transformation created (nothing executed yet)")

# ACTION (executed immediately)
# Now Spark actually executes the filter
print("\nExecuting action - show():")
filtered_df.show()

**Output:**
```
Transformation created (nothing executed yet)

Executing action - show():
+------+---+
|  name|age|
+------+---+
|   Bob| 32|
| Diana| 35|
+------+---+
```

---

## 8. Common Data Types in PySpark

| Data Type | Python Type | Description |
|-----------|-------------|-------------|
| `StringType` | str | Text/characters |
| `IntegerType` | int | Whole numbers (-2^31 to 2^31-1) |
| `LongType` | int | Large whole numbers (-2^63 to 2^63-1) |
| `DoubleType` | float | Decimal numbers |
| `FloatType` | float | Single precision decimals |
| `BooleanType` | bool | True/False |
| `DateType` | datetime.date | Date (YYYY-MM-DD) |
| `TimestampType` | datetime.datetime | Date + Time |
| `ArrayType` | list | Collection of elements |
| `MapType` | dict | Key-value pairs |
| `StructType` | - | Nested structure (columns) |

---

## 9. Mini Challenge: Test Your Understanding

### Challenge 1: Create a DataFrame
Create a DataFrame with the following data:
- Column 1: "product" (string) - ["Laptop", "Phone", "Tablet", "Monitor"]
- Column 2: "price" (integer) - [1200, 800, 400, 300]
- Column 3: "stock" (integer) - [5, 15, 20, 8]

Then:
1. Print the schema
2. Show all rows
3. Count the number of rows

### Challenge 2: Understanding Lazy Evaluation
1. Create a filter transformation (don't execute yet)
2. Create another filter transformation
3. Add a third transformation
4. Finally, call `show()` - What happened?

### Challenge 3: Examine DataFrame Properties
For the product DataFrame you created:
1. Print all column names
2. Print all data types
3. Count rows
4. Get the first row using `first()`

---

## Key Takeaways

✅ **Databricks** is a managed platform for running Spark

✅ **DBFS** is a virtual file system for storing data

✅ **SparkSession** is your entry point to Spark

✅ **DataFrames** are the main data structure in PySpark

✅ **Lazy Evaluation** means transformations aren't executed until you call an action

✅ **Transformations** are lazy, **Actions** are eager

✅ Always **print schema** to understand your data structure

---

## Next Steps
In Module 2, we'll learn how to **read and write data** in different formats (CSV, JSON, Parquet, Delta Lake)!